# FastPower (Python)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/FastPower_python.ipynb)

Chapter 14 — Recursion. Runnable notebook of `FastPower`: the algorithm in Python, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

The same notebook in [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/FastPower_cpp.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/FastPower_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion_python.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Python 3, nothing to install.

**Names.** There are no arrays in this chapter: variables and parameters have the names of the pseudo-code. Two exceptions in Python, where `from` and `next` have a special meaning: `Hanoi` uses `frm` instead of `from` and `IterativeFib` uses `nxt` instead of `next`.

## Tools: counters, stack and trace

The cell defines the counters (`cnt`, `calls`, `max_depth`), the call stack (`push` and `pop`) and the functions that print the trace (`event`, `say`) and the table of steps (`print_steps`). It has to be run only once.

In [1]:
cnt = [0] * 20      # cnt[r] = number of executions of line r (lines numbered as in the lecture notes)
calls = 0           # number of calls = nodes of the tree of recursive calls
max_depth = 0       # depth of the recursion = maximum height of the stack
mults = 0           # multiplications (FastPower)
moves = 0           # moves (Hanoi)
ev = 0              # counter of the events of the trace
stack = []          # the call stack: the arguments of the active calls
seen = [0] * 64     # seen[x] = number of calls with argument x
trace = False       # True: print the trace of the execution


def reset():
    global calls, max_depth, mults, moves, ev
    calls = max_depth = mults = moves = ev = 0
    stack.clear()
    for r in range(len(cnt)):
        cnt[r] = 0
    for k in range(len(seen)):
        seen[k] = 0


def push(x):                       # a call starts: its frame goes on top of the stack
    global calls, max_depth
    calls += 1
    stack.append(x)
    max_depth = max(max_depth, len(stack))
    if x < len(seen):
        seen[x] += 1


def pop():                         # the call returns: its frame leaves the stack
    stack.pop()


def event(text):                   # numbered line of the trace, with the stack
    global ev
    ev += 1
    print("%2d. %s   stack: %s" % (ev, text, stack))


def say(call, action):             # line of the trace: the call, indented one level for every active call
    print("%-28s| %s" % ("  " * (len(stack) - 1) + call, action))


def print_steps(*lines):
    print("Line | Executions")
    for r in lines:
        print("%4d | %10d" % (r, cnt[r]))
    print("Total steps S = %d" % sum(cnt))

## Fast exponentiation: FastPower

**Input:** a number $x$ and a natural number $n$. **Output:** $x^n$, by repeated squaring. $S(n) = 4\lfloor\log_2 n\rfloor + 6 = \Theta(\log_2 n)$ steps, $\lfloor\log_2 n\rfloor + 2$ calls, at most $2(\lfloor\log_2 n\rfloor + 1)$ multiplications.

Pseudo-code: section 14.7 of the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

In [2]:
def floor_log2(n):                                 # floor(log2 n) with a loop (only for the formula of the steps)
    k = 0
    while n > 1:
        n = n // 2
        k = k + 1
    return k


def fast_power(x, n):
    global mults
    push(n); cnt[2] += 1
    if n == 0:
        cnt[3] += 1
        if trace: say("FastPower(%d, 0)" % x, "base case, returns 1")
        pop()
        return 1
    if trace: say("FastPower(%d, %d)" % (x, n), "%d != 0 => calls FastPower(%d, %d)   (%d/2 = %d)" % (n, x, n // 2, n, n // 2))
    y = fast_power(x, n // 2); cnt[5] += 1         # one recursive call only: the result is in y
    cnt[6] += 1
    if n % 2 == 0:
        cnt[7] += 1
        mults += 1
        r = y * y
        if trace: say("FastPower(%d, %d)" % (x, n), "y = %d, %d is even => returns %d * %d = %d" % (y, n, y, y, r))
    else:
        cnt[9] += 1
        mults += 2
        r = x * y * y
        if trace: say("FastPower(%d, %d)" % (x, n), "y = %d, %d is odd => returns %d * %d * %d = %d" % (y, n, x, y, y, r))
    pop()
    return r

In [3]:
# Example of the lecture notes: FastPower(3, 13)
x, n = 3, 13
reset()
trace = True
print("%-28s| %s" % ("call", "action"))
r = fast_power(x, n)
trace = False
print("%d^%d = %d, calls = %d, multiplications = %d" % (x, n, r, calls, mults))
print_steps(2, 3, 5, 6, 7, 9)
print("S(n) = 4 floor(log2 n) + 6 = %d" % (4 * floor_log2(n) + 6))

# How many multiplications: about 2 log2 n instead of n (x = 1, to avoid overflow)
print()
print("      n |    calls | multiplications | S(n) | 4 floor(log2 n) + 6")
for n in [1, 2, 13, 100, 1000, 1000000]:
    reset()
    fast_power(1, n)
    print("%7d | %8d | %15d | %4d | %d" % (n, calls, mults, sum(cnt), 4 * floor_log2(n) + 6))

call                        | action
FastPower(3, 13)            | 13 != 0 => calls FastPower(3, 6)   (13/2 = 6)
  FastPower(3, 6)           | 6 != 0 => calls FastPower(3, 3)   (6/2 = 3)
    FastPower(3, 3)         | 3 != 0 => calls FastPower(3, 1)   (3/2 = 1)
      FastPower(3, 1)       | 1 != 0 => calls FastPower(3, 0)   (1/2 = 0)
        FastPower(3, 0)     | base case, returns 1
      FastPower(3, 1)       | y = 1, 1 is odd => returns 3 * 1 * 1 = 3
    FastPower(3, 3)         | y = 3, 3 is odd => returns 3 * 3 * 3 = 27
  FastPower(3, 6)           | y = 27, 6 is even => returns 27 * 27 = 729
FastPower(3, 13)            | y = 729, 13 is odd => returns 3 * 729 * 729 = 1594323
3^13 = 1594323, calls = 5, multiplications = 7
Line | Executions
   2 |          5
   3 |          1
   5 |          4
   6 |          4
   7 |          1
   9 |          3
Total steps S = 18
S(n) = 4 floor(log2 n) + 6 = 18

      n |    calls | multiplications | S(n) | 4 floor(log2 n) + 6
      1 |        2 |   

**Try it yourself.** Change $x$ and $n$ and count the multiplications.

In [4]:
x, n = 2, 10        # <-- change x and n (with large x and n C++ and Java overflow)
reset()
trace = True
print("%-28s| %s" % ("call", "action"))
r = fast_power(x, n)
trace = False
print("%d^%d = %d, multiplications = %d" % (x, n, r, mults))

call                        | action
FastPower(2, 10)            | 10 != 0 => calls FastPower(2, 5)   (10/2 = 5)
  FastPower(2, 5)           | 5 != 0 => calls FastPower(2, 2)   (5/2 = 2)
    FastPower(2, 2)         | 2 != 0 => calls FastPower(2, 1)   (2/2 = 1)
      FastPower(2, 1)       | 1 != 0 => calls FastPower(2, 0)   (1/2 = 0)
        FastPower(2, 0)     | base case, returns 1
      FastPower(2, 1)       | y = 1, 1 is odd => returns 2 * 1 * 1 = 2
    FastPower(2, 2)         | y = 2, 2 is even => returns 2 * 2 = 4
  FastPower(2, 5)           | y = 4, 5 is odd => returns 2 * 4 * 4 = 32
FastPower(2, 10)            | y = 32, 10 is even => returns 32 * 32 = 1024
2^10 = 1024, multiplications = 6


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).